In [0]:
dbutils.widgets.text("path", "")
config_path = dbutils.widgets.get("path")

In [0]:
import json

import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
from pyspark.sql.window import Window

from common_utils.logging import get_logger
from common_utils.generic_functions import rename_columns, safe_cast
from common_utils.scdfunctions import scd_type_1


# ============================================================
# 0. LOGGER
# ============================================================

logger = get_logger("sales_silver")


# ============================================================
# 1. READ CONFIG
# ============================================================

dbutils.widgets.text("path", "")
config_path = dbutils.widgets.get("path")

logger.info("Reading Silver configuration from %s", config_path)

with open(config_path, "r") as f:
    config = json.load(f)


# ============================================================
# 2. SOURCE / TARGET
# ============================================================

source_table = config["source"]["source_table"]
source_name = config["source"]["source_name"]
target_table = config["target"]["target_table"]


# ============================================================
# 3. SALES TRANSFORMATION
# ============================================================

def transform_sales(df, config):

    logger.info("[%s] Starting sales transformation", source_name)

    # --------------------------------------------------------
    # Rename columns
    # --------------------------------------------------------

    df = rename_columns(df,config.get("rename_columns", {}))


    # --------------------------------------------------------
    # Product JSON schema
    # --------------------------------------------------------

    product_schema = StructType([
        StructField("curr", StringType()),
        StructField("id", StringType()),
        StructField("name", StringType()),
        StructField("price", DoubleType()),
        StructField("qty", IntegerType()),
        StructField("unit", StringType())
    ])


    # --------------------------------------------------------
    # Parse and flatten product JSON
    # --------------------------------------------------------

    logger.info("[%s] Parsing product JSON", source_name)

    df = (
        df.withColumn("product", F.from_json(F.col("product"),product_schema)
        )
        .select(
            "customer_id",
            "customer_name",
            "product_name",
            "order_date",
            "product_category",
            F.col("product.curr").alias("currency"),
            F.col("product.id").alias("product_id"),
            F.col("product.name").alias("product_name_json"),
            F.col("product.price").alias("product_price"),
            F.col("product.qty").alias("quantity"),
            F.col("product.unit").alias("unit"),
            "total_price",
            "last_update_ts",
            "file_path"
        )
    )


    # --------------------------------------------------------
    # Cast columns
    # --------------------------------------------------------

    df = safe_cast( df,config.get("cast_columns", {}))


    # --------------------------------------------------------
    # Drop columns
    # --------------------------------------------------------

    drop_columns = [
        c for c in config.get("drop_columns", [])
        if c in df.columns
    ]

    if drop_columns:

        logger.info(
            "[%s] Dropping columns: %s",
            source_name,
            drop_columns
        )

        df = df.drop(*drop_columns)


    # --------------------------------------------------------
    # Deduplication
    # --------------------------------------------------------

    dedup = config.get("deduplication", {})

    if dedup.get("enabled", False):

        logger.info( "[%s] Removing duplicate records",source_name )

        key_columns = dedup["key_columns"]
        order_expressions = []

        for item in dedup["order_by"]:

            col = F.col(item["column"])

            if item.get("direction", "asc").lower() == "desc":

                col = (
                    col.desc_nulls_last()
                    if item.get("nulls", "last").lower() == "last"
                    else col.desc_nulls_first()
                )

            else:

                col = (
                    col.asc_nulls_last()
                    if item.get("nulls", "last").lower() == "last"
                    else col.asc_nulls_first()
                )

            order_expressions.append(col)

        window = (
            Window.partitionBy(*key_columns).orderBy(*order_expressions)
        )

        df = (
            df.withColumn("rn", F.row_number().over(window))
            .filter(F.col("rn") == 1)
            .drop("rn")
        )


    # --------------------------------------------------------
    # Final columns
    # --------------------------------------------------------

    logger.info("[%s] Selecting final Silver columns",source_name)
    df = df.select(*config.get("select_columns", []))
    logger.info("[%s] Sales transformation completed",source_name)

    return df


# ============================================================
# 4. READ BRONZE
# ============================================================

logger.info("[%s] Reading from Bronze table %s",source_name,source_table)

df = spark.read.table(source_table)

logger.info("[%s] Successfully read Bronze table",source_name)


# ============================================================
# 5. TRANSFORM SALES
# ============================================================

cleaned_sales = transform_sales(df,config)


# ============================================================
# 6. SCD / INITIAL LOAD
# ============================================================

if spark.catalog.tableExists(target_table):

    scd = config.get("scd", {})

    if scd.get("type") == 1:

        logger.info( "[%s] Starting SCD Type 1",source_name)

        scd_type_1(
            spark=spark,
            source_df=cleaned_sales,
            target_table=target_table,
            key_columns=scd.get("key_columns", [])
        )

    else:

        raise ValueError(
            f"Unsupported SCD type [{scd.get('type')}] "
            f"for source [{source_name}]"
        )

else:

    logger.info("[%s] Target table does not exist - creating %s",source_name,target_table)

    spark.sql("""CREATE SCHEMA IF NOT EXISTS retaildataplatform.silver""")

    (
        cleaned_sales
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

    logger.info("[%s] Silver table created successfully",source_name)


# ============================================================
# 7. COMPLETION
# ============================================================

logger.info("[%s] Bronze to Silver pipeline completed successfully", source_name)